In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

# ── Start Spark Session ────────────────────────────────────────────────────
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-sql") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# ── Load Gold Tables ───────────────────────────────────────────────────────
GOLD = "/home/jovyan/work/output/gold"

fact     = spark.read.csv(f"{GOLD}/fact_order_items", header=True, inferSchema=True)
dim_cust = spark.read.csv(f"{GOLD}/dim_customers",    header=True, inferSchema=True)
dim_prod = spark.read.csv(f"{GOLD}/dim_products",     header=True, inferSchema=True)
dim_sell = spark.read.csv(f"{GOLD}/dim_sellers",      header=True, inferSchema=True)

# Register as SQL views so we can query them with SQL syntax
fact.createOrReplaceTempView("fact_order_items")
dim_cust.createOrReplaceTempView("dim_customers")
dim_prod.createOrReplaceTempView("dim_products")
dim_sell.createOrReplaceTempView("dim_sellers")

print("✅ Gold tables loaded and registered as SQL views")
print(f"   fact_order_items : {fact.count()} rows")
print(f"   dim_customers    : {dim_cust.count()} rows")
print(f"   dim_products     : {dim_prod.count()} rows")
print(f"   dim_sellers      : {dim_sell.count()} rows")

✅ Gold tables loaded and registered as SQL views
   fact_order_items : 7662 rows
   dim_customers    : 6740 rows
   dim_products     : 32951 rows
   dim_sellers      : 3095 rows


In [2]:
# ── Query 1: Revenue Trend Analysis with Ranking ───────────────────────────
# Goal: For each product category show monthly revenue, rank within month,
# month-over-month growth %, and 3-month rolling average
# Only for top 5 categories by overall revenue
# Only months with at least 10 transactions

query_1 = """
WITH monthly AS (
    -- Step 1: Calculate monthly revenue per category
    SELECT
        p.product_category_name,
        YEAR(f.order_date)              AS year,
        MONTH(f.order_date)             AS month,
        SUM(f.price + f.freight_value)  AS monthly_revenue,
        COUNT(*)                        AS transaction_count
    FROM fact_order_items f
    JOIN dim_products p ON f.product_key = p.product_key
    GROUP BY 1, 2, 3
    HAVING COUNT(*) >= 10
),
ranked AS (
    -- Step 2: Rank categories within each month by revenue
    SELECT *,
        RANK() OVER (
            PARTITION BY year, month
            ORDER BY monthly_revenue DESC
        ) AS monthly_rank
    FROM monthly
),
top5 AS (
    -- Step 3: Find top 5 categories by OVERALL revenue
    SELECT product_category_name
    FROM monthly
    GROUP BY product_category_name
    ORDER BY SUM(monthly_revenue) DESC
    LIMIT 5
),
with_growth AS (
    -- Step 4: Calculate MoM growth and 3-month rolling average
    -- Only for top 5 categories
    SELECT r.*,
        LAG(r.monthly_revenue) OVER (
            PARTITION BY r.product_category_name
            ORDER BY r.year, r.month
        ) AS prev_month_revenue,

        AVG(r.monthly_revenue) OVER (
            PARTITION BY r.product_category_name
            ORDER BY r.year, r.month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS rolling_3m_avg_revenue

    FROM ranked r
    INNER JOIN top5 t ON r.product_category_name = t.product_category_name
)
-- Step 5: Final output with clean column names
SELECT
    product_category_name,
    year,
    month,
    ROUND(monthly_revenue, 2)                               AS monthly_revenue,
    monthly_rank,
    ROUND(
        (monthly_revenue - prev_month_revenue)
        / NULLIF(prev_month_revenue, 0) * 100
    , 2)                                                    AS mom_growth_pct,
    ROUND(rolling_3m_avg_revenue, 2)                        AS rolling_3m_avg_revenue
FROM with_growth
ORDER BY product_category_name, year, month
"""

result_q1 = spark.sql(query_1)

print("✅ Query 1 executed successfully")
print(f"   Result rows : {result_q1.count()}")
print(f"\n   Top 5 categories found:")
spark.sql("""
    SELECT product_category_name, ROUND(SUM(price + freight_value), 2) AS total_revenue
    FROM fact_order_items f
    JOIN dim_products p ON f.product_key = p.product_key
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 5
""").show()

print("\n   Sample results:")
result_q1.show(10, truncate=False)

✅ Query 1 executed successfully
   Result rows : 10

   Top 5 categories found:
+---------------------+-------------+
|product_category_name|total_revenue|
+---------------------+-------------+
|         beleza_saude|    136978.42|
|   relogios_presentes|    103724.03|
| utilidades_domest...|     83382.23|
|      cama_mesa_banho|     73883.39|
|        esporte_lazer|     70548.94|
+---------------------+-------------+


   Sample results:
+---------------------+----+-----+---------------+------------+--------------+----------------------+
|product_category_name|year|month|monthly_revenue|monthly_rank|mom_growth_pct|rolling_3m_avg_revenue|
+---------------------+----+-----+---------------+------------+--------------+----------------------+
|beleza_saude         |2018|7    |63236.86       |1           |NULL          |63236.86              |
|beleza_saude         |2018|8    |73741.56       |1           |16.61         |68489.21              |
|cama_mesa_banho      |2018|7    |34282.2      

In [3]:
# ── Save Query 1 to SQL file ───────────────────────────────────────────────
sql_dir = "/home/jovyan/work/sql"
os.makedirs(sql_dir, exist_ok=True)

query_1_sql = """-- Query 1: Revenue Trend Analysis with Ranking
-- Strategy: CTEs to build monthly revenue per category, then window
-- functions for rank, MoM growth, and 3-month rolling average.
-- Only top 5 categories by overall revenue.
-- Only months with at least 10 transactions.

WITH monthly AS (
    SELECT
        p.product_category_name,
        YEAR(f.order_date)              AS year,
        MONTH(f.order_date)             AS month,
        SUM(f.price + f.freight_value)  AS monthly_revenue,
        COUNT(*)                        AS transaction_count
    FROM fact_order_items f
    JOIN dim_products p ON f.product_key = p.product_key
    GROUP BY 1, 2, 3
    HAVING COUNT(*) >= 10
),
ranked AS (
    SELECT *,
        RANK() OVER (
            PARTITION BY year, month
            ORDER BY monthly_revenue DESC
        ) AS monthly_rank
    FROM monthly
),
top5 AS (
    SELECT product_category_name
    FROM monthly
    GROUP BY product_category_name
    ORDER BY SUM(monthly_revenue) DESC
    LIMIT 5
),
with_growth AS (
    SELECT r.*,
        LAG(r.monthly_revenue) OVER (
            PARTITION BY r.product_category_name
            ORDER BY r.year, r.month
        ) AS prev_month_revenue,
        AVG(r.monthly_revenue) OVER (
            PARTITION BY r.product_category_name
            ORDER BY r.year, r.month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS rolling_3m_avg_revenue
    FROM ranked r
    INNER JOIN top5 t ON r.product_category_name = t.product_category_name
)
SELECT
    product_category_name,
    year,
    month,
    ROUND(monthly_revenue, 2)                                AS monthly_revenue,
    monthly_rank,
    ROUND(
        (monthly_revenue - prev_month_revenue)
        / NULLIF(prev_month_revenue, 0) * 100
    , 2)                                                     AS mom_growth_pct,
    ROUND(rolling_3m_avg_revenue, 2)                         AS rolling_3m_avg_revenue
FROM with_growth
ORDER BY product_category_name, year, month
"""

with open(f"{sql_dir}/query_1.sql", "w") as f:
    f.write(query_1_sql)

print("✅ query_1.sql saved")

✅ query_1.sql saved


In [4]:
# ── Query 2: Seller Performance Scorecard ─────────────────────────────────
# Goal: Rank sellers on a composite score based on:
# - On-time delivery rate (40% weight)
# - Delivery speed (30% weight)  
# - Revenue (30% weight)
# Only sellers with at least 20 orders

query_2 = """
WITH seller_metrics AS (
    -- Step 1: Calculate raw metrics per seller
    SELECT
        s.seller_id,
        s.seller_state,
        COUNT(*)                                    AS total_orders,
        ROUND(SUM(f.price + f.freight_value), 2)   AS total_revenue,
        ROUND(
            SUM(CASE WHEN f.is_late_delivery = true THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 2
        )                                           AS late_delivery_rate,
        ROUND(AVG(f.days_delivery_vs_estimate), 2) AS avg_days_vs_estimate
    FROM fact_order_items f
    JOIN dim_sellers s ON f.seller_key = s.seller_key
    WHERE f.days_delivery_vs_estimate IS NOT NULL
    GROUP BY 1, 2
    HAVING COUNT(*) >= 20
),
percentiles AS (
    -- Step 2: Calculate percentile ranks for each metric
    -- For late_delivery_rate and avg_days_vs_estimate we INVERT
    -- because lower values are BETTER for these metrics
    SELECT *,
        ROUND(PERCENT_RANK() OVER (
            ORDER BY late_delivery_rate ASC
        ) * 100, 2)                                AS on_time_pctl,
        ROUND(PERCENT_RANK() OVER (
            ORDER BY avg_days_vs_estimate ASC
        ) * 100, 2)                                AS speed_pctl,
        ROUND(PERCENT_RANK() OVER (
            ORDER BY total_revenue DESC
        ) * 100, 2)                                AS revenue_pctl
    FROM seller_metrics
)
-- Step 3: Calculate composite score and final ranking
SELECT
    seller_id,
    seller_state,
    total_orders,
    total_revenue,
    late_delivery_rate,
    avg_days_vs_estimate,
    on_time_pctl,
    speed_pctl,
    revenue_pctl,
    ROUND(
        (on_time_pctl * 0.40) +
        (speed_pctl   * 0.30) +
        (revenue_pctl * 0.30)
    , 2)                                           AS composite_score,
    RANK() OVER (
        ORDER BY
            (on_time_pctl * 0.40) +
            (speed_pctl   * 0.30) +
            (revenue_pctl * 0.30) DESC
    )                                              AS overall_rank
FROM percentiles
ORDER BY overall_rank
"""

result_q2 = spark.sql(query_2)

print("✅ Query 2 executed successfully")
print(f"   Sellers in scorecard : {result_q2.count()}")
print(f"\n   Top 10 sellers:")
result_q2.show(10, truncate=False)

✅ Query 2 executed successfully
   Sellers in scorecard : 70

   Top 10 sellers:
+--------------------------------+------------+------------+-------------+------------------+--------------------+------------+----------+------------+---------------+------------+
|seller_id                       |seller_state|total_orders|total_revenue|late_delivery_rate|avg_days_vs_estimate|on_time_pctl|speed_pctl|revenue_pctl|composite_score|overall_rank|
+--------------------------------+------------+------------+-------------+------------------+--------------------+------------+----------+------------+---------------+------------+
|b561927807645834b59ef0d16ba55a24|SP          |25          |1387.32      |24.00             |-6.72               |100.0       |97.1      |92.75       |96.96          |1           |
|537eb890efff034a88679788b647c564|RJ          |20          |813.47       |10.00             |-7.25               |89.86       |89.86     |98.55       |92.47          |2           |
|42b729f859728

In [5]:
# ── Save Query 2 to SQL file ───────────────────────────────────────────────
query_2_sql = """-- Query 2: Seller Performance Scorecard
-- Strategy: Calculate raw metrics per seller, then use PERCENT_RANK()
-- window function for each metric. Invert ranking for late_delivery_rate
-- and avg_days_vs_estimate so lower = better becomes higher percentile.
-- Composite score = weighted average of three percentiles.
-- Only sellers with at least 20 orders included.

WITH seller_metrics AS (
    SELECT
        s.seller_id,
        s.seller_state,
        COUNT(*)                                    AS total_orders,
        ROUND(SUM(f.price + f.freight_value), 2)   AS total_revenue,
        ROUND(
            SUM(CASE WHEN f.is_late_delivery = true THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 2
        )                                           AS late_delivery_rate,
        ROUND(AVG(f.days_delivery_vs_estimate), 2) AS avg_days_vs_estimate
    FROM fact_order_items f
    JOIN dim_sellers s ON f.seller_key = s.seller_key
    WHERE f.days_delivery_vs_estimate IS NOT NULL
    GROUP BY 1, 2
    HAVING COUNT(*) >= 20
),
percentiles AS (
    SELECT *,
        ROUND(PERCENT_RANK() OVER (
            ORDER BY late_delivery_rate ASC
        ) * 100, 2)                                AS on_time_pctl,
        ROUND(PERCENT_RANK() OVER (
            ORDER BY avg_days_vs_estimate ASC
        ) * 100, 2)                                AS speed_pctl,
        ROUND(PERCENT_RANK() OVER (
            ORDER BY total_revenue DESC
        ) * 100, 2)                                AS revenue_pctl
    FROM seller_metrics
)
SELECT
    seller_id,
    seller_state,
    total_orders,
    total_revenue,
    late_delivery_rate,
    avg_days_vs_estimate,
    on_time_pctl,
    speed_pctl,
    revenue_pctl,
    ROUND(
        (on_time_pctl * 0.40) +
        (speed_pctl   * 0.30) +
        (revenue_pctl * 0.30)
    , 2)                                           AS composite_score,
    RANK() OVER (
        ORDER BY
            (on_time_pctl * 0.40) +
            (speed_pctl   * 0.30) +
            (revenue_pctl * 0.30) DESC
    )                                              AS overall_rank
FROM percentiles
ORDER BY overall_rank
"""

with open(f"{sql_dir}/query_2.sql", "w") as f:
    f.write(query_2_sql)

print("✅ query_2.sql saved")
print(f"\n📁 SQL files created:")
for f in os.listdir(sql_dir):
    print(f"   {f}")

✅ query_2.sql saved

📁 SQL files created:
   query_1.sql
   query_2.sql
